# ZMART target acquisition — v4 (controller-only)

This notebook drives the microscope **through the `zmart_controller` Session
surface only** — no `navigator_expert` calls in the operator flow. Each step is
a 1–3 line invocation of a `pipeline` function; the driver does the work.

Flow: **connect → collect state → load positions → focus → overview →
discover targets → acquire targets → summary.**

The driver adapter is imported once (its import registers the instrument with
`zmart_controller`); after that the workflow never touches it directly.

## 0 · Setup — paths & config
Set the paths and run parameters, then import `pipeline`.

In [ ]:
import sys
from pathlib import Path

# --- paths (edit REPO_ROOT if this notebook is moved) ---
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "zmart_controller").is_dir():
    REPO_ROOT = REPO_ROOT.parent
DRIVER_PARENT = REPO_ROOT / "zmart_drivers" / "leica" / "stellaris5_y42h93"
TARGET_ACQ = REPO_ROOT / "workflows" / "target_acquisition"
for p in (str(DRIVER_PARENT), str(REPO_ROOT), str(TARGET_ACQ)):
    if p not in sys.path:
        sys.path.insert(0, p)

# --- run config ---
VENDOR = "leica"
OUTPUT_ROOT = Path("D:/zmart-output")          # where run folders are written
ANALYSIS_REPO = Path("C:/code/smart-analysis")  # cellpose analysis engine repo
POSITIONS_JSON = TARGET_ACQ / "positions.json"  # [{"x":..,"y":..}, ...] frame um

# overview job geometry (one value for all tiles — read from LAS X / the saved image)
PIXEL_SIZE_UM = 0.5
IMAGE_SIZE_PX = (2048, 2048)   # (H, W)
AF_JOB = None                  # autofocus job name (None if the instrument has one)

# focus points: where to autofocus (frame um). Fewer than the overview tiles.
FOCUS_POINTS = [{"x": 0.0, "y": 0.0}, {"x": 5000.0, "y": 0.0}, {"x": 0.0, "y": 5000.0}]

import pipeline  # noqa: E402  (paths are set just above, in this cell)
print("pipeline surface:", ", ".join(pipeline.__all__))

## 1 · Connect

Importing the adapter registers the Leica instrument; `pipeline.connect` opens
the controller session. `set_origin()` makes the current stage position the
frame origin — every position below is micrometres from here.

In [ ]:
import navigator_expert.zmart_adapter  # noqa: F401 — import registers the instrument

mic = pipeline.connect(VENDOR, output_root=OUTPUT_ROOT)
mic.set_origin()
mic.context

## 2 · Collect state(s)

A "state" is a selected LAS X job + its settings. Capture one per role. The
overview state is applied once before the overview run; the target state before
the target run. `set_state` reapplies the *changeable* part only.

In [ ]:
# Select the OVERVIEW job in LAS X, then run this cell:
overview_state = mic.get_state()

# Select the TARGET job in LAS X, then run this cell:
target_state = mic.get_state()
overview_state["observed"]

## 3 · Load positions
The overview tile centres, as frame micrometres.

In [ ]:
positions = pipeline.load_positions(POSITIONS_JSON)
print(len(positions), "overview positions")

## 4 · Focus

Autofocus at the chosen points (`run_procedure('autofocus')` under the hood),
fit a surface z(x, y), and preview it. `run_overview` / `acquire_targets` query
this surface for each position's z.

In [ ]:
measured = pipeline.measure_focus(mic, FOCUS_POINTS, af_job=AF_JOB)
focus = pipeline.fit_focus_surface(measured)
pipeline.plot_focus_surface(focus, save_path=OUTPUT_ROOT / "focus_surface.png", show=True)
print("focus model:", focus.model)

## 5 · Overview

Apply the overview state, then capture at every position (z from the surface).
Returns the driver's per-position acquire records.

In [ ]:
overview_records = pipeline.run_overview(mic, positions, state=overview_state, focus=focus)
print(len(overview_records), "overviews captured")

## 6 · Discover targets

Segment each overview with the cellpose analysis engine and convert each cell
centroid to a frame target. `build_overview_inputs` pairs the positions we
captured at with the saved image paths (the Leica record carries `images`); the
engine is the external smart-analysis `Engine` (`submit`/`status`/`results`).

In [ ]:
# analysis engine from the configured repo (submit/status/results contract)
if str(ANALYSIS_REPO) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_REPO))
from engine import Engine  # noqa: E402

engine = Engine()

placed = pipeline.with_focus_z(positions, focus)
image_paths = [rec["images"][0] for rec in overview_records]  # Leica record shape
overviews = pipeline.build_overview_inputs(
    placed, image_paths, pixel_size_um=PIXEL_SIZE_UM, image_size_px=IMAGE_SIZE_PX
)
targets = pipeline.discover_targets(engine, overviews)
print(len(targets), "targets discovered")

## 7 · Acquire targets
Apply the target state and acquire at each discovered position.

In [ ]:
target_records = pipeline.acquire_targets(mic, targets, state=target_state, focus=focus)
print(len(target_records), "targets acquired")

## 8 · Summary
Write a JSON summary and a frame-layout map of the run.

In [ ]:
summary = pipeline.summarize_run(
    focus=focus,
    overview_positions=placed,
    overview_records=overview_records,
    targets=targets,
)
pipeline.write_summary(summary, OUTPUT_ROOT / "summary.json")
pipeline.plot_frame_layout(
    overview_positions=placed, targets=targets, focus=focus,
    save_path=OUTPUT_ROOT / "run_layout.png", show=True,
)
summary

In [ ]:
mic.disconnect()